# Emissions Sensitivity Analysis

Breakdown of GHG emissions across sensitivity analyses.

**Figure 1** (emissions overview):
- Row 1: Net emissions by gas (CO₂, CH₄, N₂O in CO₂eq terms) — stacked area
- Rows 2–4: Per-gas breakdown by emission source — stacked area

**Figure 2** (land use change emissions):
- Row 1: LUC emissions by continent (M49 regions)
- Row 2: LUC emissions by land type (cropland/pasture expansion, cropland/grassland sparing)

**Figure 3** (land use change area):
- Row 1: Land use change area by continent (Mha)
- Row 2: Land use change area by land type (Mha)

**Figure 4** (feed breakdown):
- Row 1: Feed use by category (Gt DM)
- Row 2: Feed use by animal type (Gt DM)

In [ ]:
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
from sensitivity_utils import (
    ANIMAL_COLORS,
    CONTINENT_ORDER,
    CROP_GROUP_COLORS,
    FEED_COLORS,
    FEED_ORDER,
    FONTSIZE_AXIS_LABEL,
    FONTSIZE_PANEL_LABEL,
    FONTSIZE_TICK_LABEL,
    FONTSIZE_TITLE,
    GAS_CMAPS,
    GAS_DISPLAY,
    LUC_TYPE_DISPLAY,
    PRETTY_NAMES_EMISSIONS,
    PRETTY_NAMES_GAS,
    aggregate_crop_metrics_by_group,
    extract_combined_scenarios,
    extract_scenarios_with_param,
    get_log_ticks,
    load_crop_yield_and_production,
    load_emissions_by_source,
    load_feed_breakdown,
    load_luc_breakdown,
    load_net_emissions_by_gas,
    log_scale_zero_position,
    plot_stacked_emissions,
    set_dual_xaxis_labels,
    set_dual_xlabel,
)

In [ ]:
# Configuration
PROJECT_ROOT = Path("..").resolve()
CONFIG_NAME = "sensitivity"

## Extract scenarios

In [ ]:
# YLL scenarios
yll_scenarios_all = extract_scenarios_with_param(
    PROJECT_ROOT,
    CONFIG_NAME,
    param_path=["health", "value_per_yll"],
    scenario_prefix="yll_",
)
yll_param_values_full = [p for p, _, _ in yll_scenarios_all]
yll_scenarios = [(p, s, f) for p, s, f in yll_scenarios_all if f.exists()]
print(f"YLL: {len(yll_scenarios)}/{len(yll_scenarios_all)} solved")

# GHG scenarios
ghg_scenarios_all = extract_scenarios_with_param(
    PROJECT_ROOT,
    CONFIG_NAME,
    param_path=["emissions", "ghg_price"],
    scenario_prefix="ghg_",
)
ghg_scenarios_all = [
    (p, s, f) for p, s, f in ghg_scenarios_all if not s.startswith("ghg_yll_")
]
ghg_param_values_full = [p for p, _, _ in ghg_scenarios_all]
ghg_scenarios = [(p, s, f) for p, s, f in ghg_scenarios_all if f.exists()]
print(f"GHG: {len(ghg_scenarios)}/{len(ghg_scenarios_all)} solved")

# Combined GHG+YLL scenarios
ghg_yll_scenarios_all = extract_combined_scenarios(
    PROJECT_ROOT,
    CONFIG_NAME,
    ghg_param_path=["emissions", "ghg_price"],
    yll_param_path=["health", "value_per_yll"],
    scenario_prefix="ghg_yll_",
)
ghg_yll_ghg_values_full = [ghg for ghg, _, _, _ in ghg_yll_scenarios_all]
ghg_yll_yll_values_full = [yll for _, yll, _, _ in ghg_yll_scenarios_all]
ghg_yll_scenarios_full = [
    (ghg, yll, s, f)
    for ghg, yll, s, f in ghg_yll_scenarios_all
    if f.exists() and "break" not in s
]
ghg_yll_scenarios = [(ghg, s, f) for ghg, yll, s, f in ghg_yll_scenarios_full]
ghg_yll_param_pairs = [(ghg, yll) for ghg, yll, s, f in ghg_yll_scenarios_full]
ghg_yll_ghg_values = [ghg for ghg, _ in ghg_yll_param_pairs]
ghg_yll_yll_values = [yll for _, yll in ghg_yll_param_pairs]
print(f"GHG+YLL: {len(ghg_yll_scenarios)}/{len(ghg_yll_scenarios_all)} solved")

## Load data

In [ ]:
# Net emissions by gas
df_yll_gas = load_net_emissions_by_gas(
    yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "yll_value"
)
df_ghg_gas = load_net_emissions_by_gas(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price"
)
df_ghg_yll_gas = load_net_emissions_by_gas(
    ghg_yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price"
)

print(f"YLL by-gas shape: {df_yll_gas.shape}")
print(df_yll_gas)
print()
print(f"GHG by-gas shape: {df_ghg_gas.shape}")
print(df_ghg_gas)
print()
print(f"Combined by-gas shape: {df_ghg_yll_gas.shape}")
print(df_ghg_yll_gas)

In [ ]:
# Per-source breakdown from analysis CSVs
yll_by_source = load_emissions_by_source(
    yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "yll_value"
)
ghg_by_source = load_emissions_by_source(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price"
)
ghg_yll_by_source = load_emissions_by_source(
    ghg_yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price"
)

for label, by_source in [
    ("YLL", yll_by_source),
    ("GHG", ghg_by_source),
    ("Combined", ghg_yll_by_source),
]:
    for gas in ["CO2", "CH4", "N2O"]:
        cols = list(by_source[gas].columns) if not by_source[gas].empty else []
        print(f"{label} {gas}: {cols}")

In [ ]:
# LUC breakdown by continent (emissions)
yll_luc_continent = load_luc_breakdown(
    yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "yll_value", groupby="continent"
)
ghg_luc_continent = load_luc_breakdown(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price", groupby="continent"
)
ghg_yll_luc_continent = load_luc_breakdown(
    ghg_yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price", groupby="continent"
)

# LUC breakdown by land type (emissions)
yll_luc_type = load_luc_breakdown(
    yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "yll_value", groupby="land_type"
)
ghg_luc_type = load_luc_breakdown(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price", groupby="land_type"
)
ghg_yll_luc_type = load_luc_breakdown(
    ghg_yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price", groupby="land_type"
)

# LUC breakdown by continent (area)
yll_luc_continent_area = load_luc_breakdown(
    yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "yll_value",
    groupby="continent",
    quantity="area",
)
ghg_luc_continent_area = load_luc_breakdown(
    ghg_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "ghg_price",
    groupby="continent",
    quantity="area",
)
ghg_yll_luc_continent_area = load_luc_breakdown(
    ghg_yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "ghg_price",
    groupby="continent",
    quantity="area",
)

# LUC breakdown by land type (area)
yll_luc_type_area = load_luc_breakdown(
    yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "yll_value",
    groupby="land_type",
    quantity="area",
)
ghg_luc_type_area = load_luc_breakdown(
    ghg_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "ghg_price",
    groupby="land_type",
    quantity="area",
)
ghg_yll_luc_type_area = load_luc_breakdown(
    ghg_yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "ghg_price",
    groupby="land_type",
    quantity="area",
)

for label, df in [
    ("YLL", yll_luc_continent),
    ("GHG", ghg_luc_continent),
    ("Combined", ghg_yll_luc_continent),
]:
    print(f"{label} LUC by continent: {list(df.columns)}")
for label, df in [
    ("YLL", yll_luc_type),
    ("GHG", ghg_luc_type),
    ("Combined", ghg_yll_luc_type),
]:
    print(f"{label} LUC by type: {list(df.columns)}")

In [ ]:
# Feed breakdown by category
yll_feed_cat = load_feed_breakdown(
    yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "yll_value", groupby="feed_category"
)
ghg_feed_cat = load_feed_breakdown(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price", groupby="feed_category"
)
ghg_yll_feed_cat = load_feed_breakdown(
    ghg_yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price", groupby="feed_category"
)

# Feed breakdown by animal
yll_feed_animal = load_feed_breakdown(
    yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "yll_value", groupby="animal"
)
ghg_feed_animal = load_feed_breakdown(
    ghg_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price", groupby="animal"
)
ghg_yll_feed_animal = load_feed_breakdown(
    ghg_yll_scenarios, PROJECT_ROOT, CONFIG_NAME, "ghg_price", groupby="animal"
)

for label, df in [
    ("YLL", yll_feed_cat),
    ("GHG", ghg_feed_cat),
    ("Combined", ghg_yll_feed_cat),
]:
    print(f"{label} feed by category: {list(df.columns)}")
for label, df in [
    ("YLL", yll_feed_animal),
    ("GHG", ghg_feed_animal),
    ("Combined", ghg_yll_feed_animal),
]:
    print(f"{label} feed by animal: {list(df.columns)}")

## Emissions sensitivity figure

In [ ]:
# Restore inline backend (emissions loader imports plotting modules that set PDF backend)
from IPython import get_ipython

_ip = get_ipython()
if _ip is not None:
    _ip.run_line_magic("matplotlib", "inline")

plt.switch_backend("module://matplotlib_inline.backend_inline")
print(f"Matplotlib backend: {matplotlib.get_backend()}")

# X-axis configuration
YLL_XTICKS, YLL_XTICKLABELS = get_log_ticks(yll_param_values_full)
YLL_XLABEL = "Value per Year of Life Lost [USD/YLL]"

GHG_XTICKS, GHG_XTICKLABELS = get_log_ticks(ghg_param_values_full)
GHG_XLABEL = "GHG price [USD/tCO\u2082eq]"

GHG_YLL_XTICKS, GHG_YLL_XTICKLABELS = get_log_ticks(ghg_yll_ghg_values_full)
tick_to_yll = dict(zip(ghg_yll_ghg_values_full, ghg_yll_yll_values_full))
tick_to_yll.update(dict(zip(ghg_yll_ghg_values, ghg_yll_yll_values)))
GHG_YLL_GHG_VALUES = list(GHG_YLL_XTICKS)
GHG_YLL_YLL_VALUES = []
for tick, label in zip(GHG_YLL_XTICKS, GHG_YLL_XTICKLABELS):
    if label == "0":
        GHG_YLL_YLL_VALUES.append(0.0)
    elif tick in tick_to_yll:
        GHG_YLL_YLL_VALUES.append(tick_to_yll[tick])
    else:
        nonzero_pairs = [
            (g, y)
            for g, y in zip(ghg_yll_ghg_values_full, ghg_yll_yll_values_full)
            if g > 0
        ]
        ratio = nonzero_pairs[0][1] / nonzero_pairs[0][0] if nonzero_pairs else 1
        GHG_YLL_YLL_VALUES.append(tick * ratio)


# --- Colour assignments ---


def assign_gas_colors(df):
    """Assign colours to gas columns."""
    gas_colors = {
        GAS_DISPLAY["co2"]: "#636363",
        GAS_DISPLAY["ch4"]: "#31a354",
        GAS_DISPLAY["n2o"]: "#e6550d",
    }
    return {col: gas_colors.get(col, "#999999") for col in df.columns}


def assign_source_colors_for_gas(gas, all_sources):
    """Assign gas-specific colormap shades to source categories."""
    cmap = matplotlib.colormaps[GAS_CMAPS.get(gas, "Blues")]
    n = len(all_sources)
    colors = {}
    for i, src in enumerate(all_sources):
        shade = 0.3 + 0.55 * i / max(n - 1, 1)
        colors[src] = cmap(shade)
    return colors


def prepare_source_df(df, threshold=0.005):
    """Order columns by mean absolute value (largest first), drop near-zero."""
    if df.empty:
        return df
    significant = df.columns[df.abs().max() >= threshold]
    df = df[significant]
    # Separate positive-mean and negative-mean columns
    pos_cols = sorted(
        [c for c in df.columns if df[c].mean() >= 0],
        key=lambda c: df[c].mean(),
        reverse=True,
    )
    neg_cols = sorted(
        [c for c in df.columns if df[c].mean() < 0],
        key=lambda c: df[c].mean(),
    )
    return df[pos_cols + neg_cols]


# --- Prepare data ---
gas_colors = assign_gas_colors(df_yll_gas)

# Order gas columns: largest first
gas_order = df_yll_gas.mean().sort_values(ascending=False).index.tolist()
df_yll_gas_plot = df_yll_gas[gas_order]
df_ghg_gas_plot = df_ghg_gas[[c for c in gas_order if c in df_ghg_gas.columns]]
df_ghg_yll_gas_plot = df_ghg_yll_gas[
    [c for c in gas_order if c in df_ghg_yll_gas.columns]
]

# Prepare source DataFrames and build consistent colour maps per gas
source_dfs = {}
source_colors = {}
for gas in ["CO2", "CH4", "N2O"]:
    # Collect all source names across the three sensitivity runs
    all_sources_for_gas = set()
    for label, by_source in [
        ("yll", yll_by_source),
        ("ghg", ghg_by_source),
        ("ghg_yll", ghg_yll_by_source),
    ]:
        df_src = prepare_source_df(by_source[gas])
        source_dfs[(gas, label)] = df_src
        all_sources_for_gas.update(df_src.columns.tolist())
    source_colors[gas] = assign_source_colors_for_gas(gas, sorted(all_sources_for_gas))

# --- Build figure: 4 rows x 3 columns ---
fig, axes = plt.subplots(4, 3, figsize=(8, 9))

gases_for_rows = ["CO2", "CH4", "N2O"]
panel_labels = "abcdefghijkl"
panel_idx = 0

# --- Row 0: Net emissions by gas ---
for col, (df_gas, xlabel, x_ticks, x_ticklabels) in enumerate(
    [
        (df_yll_gas_plot, YLL_XLABEL, YLL_XTICKS, YLL_XTICKLABELS),
        (df_ghg_gas_plot, GHG_XLABEL, GHG_XTICKS, GHG_XTICKLABELS),
        (
            df_ghg_yll_gas_plot,
            "",
            GHG_YLL_XTICKS,
            [""] * len(GHG_YLL_XTICKS),
        ),
    ]
):
    if df_gas.empty:
        panel_idx += 1
        continue
    plot_stacked_emissions(
        df_gas,
        gas_colors,
        axes[0, col],
        xlabel=xlabel,
        ylabel="Net emissions [GtCO\u2082eq]" if col == 0 else "",
        panel_label=panel_labels[panel_idx],
        x_ticks=x_ticks,
        x_ticklabels=x_ticklabels,
        min_height_for_label=0.5,
        pretty_names=PRETTY_NAMES_GAS,
    )
    panel_idx += 1

# --- Rows 1-3: Per-gas source breakdown ---
for row_offset, gas in enumerate(gases_for_rows):
    row = row_offset + 1
    gas_display = GAS_DISPLAY[gas]
    for col, (label, xlabel, x_ticks, x_ticklabels) in enumerate(
        [
            ("yll", YLL_XLABEL, YLL_XTICKS, YLL_XTICKLABELS),
            ("ghg", GHG_XLABEL, GHG_XTICKS, GHG_XTICKLABELS),
            ("ghg_yll", "", GHG_YLL_XTICKS, [""] * len(GHG_YLL_XTICKS)),
        ]
    ):
        df_src = source_dfs[(gas, label)]
        if df_src.empty:
            panel_idx += 1
            continue
        colors_for_panel = {
            src: source_colors[gas][src]
            for src in df_src.columns
            if src in source_colors[gas]
        }
        plot_stacked_emissions(
            df_src,
            colors_for_panel,
            axes[row, col],
            xlabel=xlabel,
            ylabel=f"{gas_display} emissions [GtCO\u2082eq]" if col == 0 else "",
            panel_label=panel_labels[panel_idx],
            x_ticks=x_ticks,
            x_ticklabels=x_ticklabels,
            min_height_for_label=0.08,
            pretty_names=PRETTY_NAMES_EMISSIONS,
        )
        panel_idx += 1

# Column titles
axes[0, 0].set_title(
    "Health value sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)
axes[0, 1].set_title(
    "GHG price sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)
axes[0, 2].set_title(
    "Combined sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)

# Share y-axis within rows
for row in range(4):
    y_min = min(ax.get_ylim()[0] for ax in axes[row, :])
    y_max = max(ax.get_ylim()[1] for ax in axes[row, :])
    for col in range(3):
        axes[row, col].set_ylim(y_min, y_max)

# Remove x-labels and ticks except bottom row
for row in range(3):
    for col in range(3):
        axes[row, col].set_xlabel("")
        axes[row, col].set_xticklabels([])

# Dual x-axis labels on bottom-right
set_dual_xaxis_labels(
    axes[3, 2], GHG_YLL_XTICKS, GHG_YLL_GHG_VALUES, GHG_YLL_YLL_VALUES
)
set_dual_xlabel(axes[3, 2])

# Remove y-labels from cols 1 and 2
for row in range(4):
    for col in [1, 2]:
        axes[row, col].set_ylabel("")
        axes[row, col].set_yticklabels([])

fig.align_ylabels(axes[:, 0])
plt.tight_layout()
plt.subplots_adjust(bottom=0.08)

try:
    from IPython.display import display

    display(fig)
except Exception:
    plt.show()
plt.close(fig)

## Land use change figure

In [ ]:
# --- Colour assignments for LUC ---

# Continent colours (qualitative palette)
CONTINENT_COLORS = {
    "Africa": "#e41a1c",
    "Americas": "#377eb8",
    "Asia": "#4daf4a",
    "Europe": "#984ea3",
    "Oceania": "#ff7f00",
    "Other": "#999999",
}

# LUC type colours: warm for expansion (positive), cool for sparing (negative)
LUC_TYPE_COLORS = {
    "Cropland expansion": "#d62728",
    "Pasture expansion": "#ff7f0e",
    "Cropland sparing": "#1f77b4",
    "Grassland sparing (convertible)": "#2ca02c",
    "Grassland sparing (marginal)": "#98df8a",
}


def prepare_luc_df(df, col_order=None, threshold=0.005):
    """Prepare LUC DataFrame: reorder columns and drop near-zero."""
    if df.empty:
        return df
    significant = df.columns[df.abs().max() >= threshold]
    df = df[significant]
    if col_order is not None:
        ordered = [c for c in col_order if c in df.columns]
        extra = [c for c in df.columns if c not in ordered]
        df = df[ordered + extra]
    return df


# --- Prepare LUC data ---
luc_continent_dfs = {}
luc_type_dfs = {}

continent_col_order = CONTINENT_ORDER
luc_type_col_order = list(LUC_TYPE_COLORS.keys())

for label, df_cont, df_type in [
    ("yll", yll_luc_continent, yll_luc_type),
    ("ghg", ghg_luc_continent, ghg_luc_type),
    ("ghg_yll", ghg_yll_luc_continent, ghg_yll_luc_type),
]:
    luc_continent_dfs[label] = prepare_luc_df(df_cont, continent_col_order)
    luc_type_dfs[label] = prepare_luc_df(df_type, luc_type_col_order)

# --- Build figure: 2 rows x 3 columns ---
fig, axes = plt.subplots(2, 3, figsize=(8, 4.5))

panel_labels = "abcdef"
panel_idx = 0

# --- Row 0: LUC by continent ---
for col, (label, xlabel, x_ticks, x_ticklabels) in enumerate(
    [
        ("yll", YLL_XLABEL, YLL_XTICKS, YLL_XTICKLABELS),
        ("ghg", GHG_XLABEL, GHG_XTICKS, GHG_XTICKLABELS),
        ("ghg_yll", "", GHG_YLL_XTICKS, [""] * len(GHG_YLL_XTICKS)),
    ]
):
    df = luc_continent_dfs[label]
    if df.empty:
        panel_idx += 1
        continue
    colors = {c: CONTINENT_COLORS.get(c, "#999999") for c in df.columns}
    plot_stacked_emissions(
        df,
        colors,
        axes[0, col],
        xlabel=xlabel,
        ylabel="LUC emissions [GtCO\u2082]" if col == 0 else "",
        panel_label=panel_labels[panel_idx],
        x_ticks=x_ticks,
        x_ticklabels=x_ticklabels,
        min_height_for_label=0.3,
    )
    panel_idx += 1

# --- Row 1: LUC by land type ---
for col, (label, xlabel, x_ticks, x_ticklabels) in enumerate(
    [
        ("yll", YLL_XLABEL, YLL_XTICKS, YLL_XTICKLABELS),
        ("ghg", GHG_XLABEL, GHG_XTICKS, GHG_XTICKLABELS),
        ("ghg_yll", "", GHG_YLL_XTICKS, [""] * len(GHG_YLL_XTICKS)),
    ]
):
    df = luc_type_dfs[label]
    if df.empty:
        panel_idx += 1
        continue
    colors = {c: LUC_TYPE_COLORS.get(c, "#999999") for c in df.columns}
    plot_stacked_emissions(
        df,
        colors,
        axes[1, col],
        xlabel=xlabel,
        ylabel="LUC emissions [GtCO\u2082]" if col == 0 else "",
        panel_label=panel_labels[panel_idx],
        x_ticks=x_ticks,
        x_ticklabels=x_ticklabels,
        min_height_for_label=0.3,
        pretty_names=LUC_TYPE_DISPLAY,
    )
    panel_idx += 1

# Column titles
axes[0, 0].set_title(
    "Health value sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)
axes[0, 1].set_title(
    "GHG price sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)
axes[0, 2].set_title(
    "Combined sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)

# Share y-axis within rows
for row in range(2):
    y_min = min(ax.get_ylim()[0] for ax in axes[row, :])
    y_max = max(ax.get_ylim()[1] for ax in axes[row, :])
    for col in range(3):
        axes[row, col].set_ylim(y_min, y_max)

# Remove x-labels and ticks from top row
for col in range(3):
    axes[0, col].set_xlabel("")
    axes[0, col].set_xticklabels([])

# Dual x-axis labels on bottom-right
set_dual_xaxis_labels(
    axes[1, 2], GHG_YLL_XTICKS, GHG_YLL_GHG_VALUES, GHG_YLL_YLL_VALUES
)
set_dual_xlabel(axes[1, 2])

# Remove y-labels from cols 1 and 2
for row in range(2):
    for col in [1, 2]:
        axes[row, col].set_ylabel("")
        axes[row, col].set_yticklabels([])

fig.align_ylabels(axes[:, 0])
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)

try:
    from IPython.display import display

    display(fig)
except Exception:
    plt.show()
plt.close(fig)

## Land use change area figure

In [ ]:
# --- Prepare LUC area data ---
luc_continent_area_dfs = {}
luc_type_area_dfs = {}

for label, df_cont, df_type in [
    ("yll", yll_luc_continent_area, yll_luc_type_area),
    ("ghg", ghg_luc_continent_area, ghg_luc_type_area),
    ("ghg_yll", ghg_yll_luc_continent_area, ghg_yll_luc_type_area),
]:
    luc_continent_area_dfs[label] = prepare_luc_df(df_cont, continent_col_order)
    luc_type_area_dfs[label] = prepare_luc_df(df_type, luc_type_col_order)

# --- Build figure: 2 rows x 3 columns ---
fig, axes = plt.subplots(2, 3, figsize=(8, 4.5))

panel_labels = "abcdef"
panel_idx = 0

# --- Row 0: LUC area by continent ---
for col, (label, xlabel, x_ticks, x_ticklabels) in enumerate(
    [
        ("yll", YLL_XLABEL, YLL_XTICKS, YLL_XTICKLABELS),
        ("ghg", GHG_XLABEL, GHG_XTICKS, GHG_XTICKLABELS),
        ("ghg_yll", "", GHG_YLL_XTICKS, [""] * len(GHG_YLL_XTICKS)),
    ]
):
    df = luc_continent_area_dfs[label]
    if df.empty:
        panel_idx += 1
        continue
    colors = {c: CONTINENT_COLORS.get(c, "#999999") for c in df.columns}
    plot_stacked_emissions(
        df,
        colors,
        axes[0, col],
        xlabel=xlabel,
        ylabel="Land use change [Mha]" if col == 0 else "",
        panel_label=panel_labels[panel_idx],
        x_ticks=x_ticks,
        x_ticklabels=x_ticklabels,
        min_height_for_label=30,
    )
    panel_idx += 1

# --- Row 1: LUC area by land type ---
for col, (label, xlabel, x_ticks, x_ticklabels) in enumerate(
    [
        ("yll", YLL_XLABEL, YLL_XTICKS, YLL_XTICKLABELS),
        ("ghg", GHG_XLABEL, GHG_XTICKS, GHG_XTICKLABELS),
        ("ghg_yll", "", GHG_YLL_XTICKS, [""] * len(GHG_YLL_XTICKS)),
    ]
):
    df = luc_type_area_dfs[label]
    if df.empty:
        panel_idx += 1
        continue
    colors = {c: LUC_TYPE_COLORS.get(c, "#999999") for c in df.columns}
    plot_stacked_emissions(
        df,
        colors,
        axes[1, col],
        xlabel=xlabel,
        ylabel="Land use change [Mha]" if col == 0 else "",
        panel_label=panel_labels[panel_idx],
        x_ticks=x_ticks,
        x_ticklabels=x_ticklabels,
        min_height_for_label=30,
        pretty_names=LUC_TYPE_DISPLAY,
    )
    panel_idx += 1

# Column titles
axes[0, 0].set_title(
    "Health value sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)
axes[0, 1].set_title(
    "GHG price sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)
axes[0, 2].set_title(
    "Combined sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)

# Share y-axis within rows
for row in range(2):
    y_min = min(ax.get_ylim()[0] for ax in axes[row, :])
    y_max = max(ax.get_ylim()[1] for ax in axes[row, :])
    for col in range(3):
        axes[row, col].set_ylim(y_min, y_max)

# Remove x-labels and ticks from top row
for col in range(3):
    axes[0, col].set_xlabel("")
    axes[0, col].set_xticklabels([])

# Dual x-axis labels on bottom-right
set_dual_xaxis_labels(
    axes[1, 2], GHG_YLL_XTICKS, GHG_YLL_GHG_VALUES, GHG_YLL_YLL_VALUES
)
set_dual_xlabel(axes[1, 2])

# Remove y-labels from cols 1 and 2
for row in range(2):
    for col in [1, 2]:
        axes[row, col].set_ylabel("")
        axes[row, col].set_yticklabels([])

fig.align_ylabels(axes[:, 0])
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)

try:
    from IPython.display import display

    display(fig)
except Exception:
    plt.show()
plt.close(fig)

## Feed breakdown figure

In [ ]:
# --- Prepare feed data ---
feed_cat_dfs = {}
feed_animal_dfs = {}

# Sort animals by mean total feed (descending) across all scenarios
all_animals = set()
for df in [yll_feed_animal, ghg_feed_animal, ghg_yll_feed_animal]:
    if not df.empty:
        all_animals.update(df.columns)
animal_means = {}
for animal in all_animals:
    vals = []
    for df in [yll_feed_animal, ghg_feed_animal, ghg_yll_feed_animal]:
        if animal in df.columns:
            vals.append(df[animal].mean())
    animal_means[animal] = np.mean(vals) if vals else 0
animal_col_order = sorted(animal_means, key=lambda a: animal_means[a], reverse=True)

for label, df_cat, df_animal in [
    ("yll", yll_feed_cat, yll_feed_animal),
    ("ghg", ghg_feed_cat, ghg_feed_animal),
    ("ghg_yll", ghg_yll_feed_cat, ghg_yll_feed_animal),
]:
    feed_cat_dfs[label] = prepare_luc_df(df_cat, FEED_ORDER)
    feed_animal_dfs[label] = prepare_luc_df(df_animal, animal_col_order)

# --- Build figure: 2 rows x 3 columns ---
fig, axes = plt.subplots(2, 3, figsize=(8, 4.5))

panel_labels = "abcdef"
panel_idx = 0

# --- Row 0: Feed by category ---
for col, (label, xlabel, x_ticks, x_ticklabels) in enumerate(
    [
        ("yll", YLL_XLABEL, YLL_XTICKS, YLL_XTICKLABELS),
        ("ghg", GHG_XLABEL, GHG_XTICKS, GHG_XTICKLABELS),
        ("ghg_yll", "", GHG_YLL_XTICKS, [""] * len(GHG_YLL_XTICKS)),
    ]
):
    df = feed_cat_dfs[label]
    if df.empty:
        panel_idx += 1
        continue
    colors = {c: FEED_COLORS.get(c, "#999999") for c in df.columns}
    plot_stacked_emissions(
        df,
        colors,
        axes[0, col],
        xlabel=xlabel,
        ylabel="Feed use [Gt DM]" if col == 0 else "",
        panel_label=panel_labels[panel_idx],
        x_ticks=x_ticks,
        x_ticklabels=x_ticklabels,
        min_height_for_label=0.3,
    )
    panel_idx += 1

# --- Row 1: Feed by animal ---
for col, (label, xlabel, x_ticks, x_ticklabels) in enumerate(
    [
        ("yll", YLL_XLABEL, YLL_XTICKS, YLL_XTICKLABELS),
        ("ghg", GHG_XLABEL, GHG_XTICKS, GHG_XTICKLABELS),
        ("ghg_yll", "", GHG_YLL_XTICKS, [""] * len(GHG_YLL_XTICKS)),
    ]
):
    df = feed_animal_dfs[label]
    if df.empty:
        panel_idx += 1
        continue
    colors = {c: ANIMAL_COLORS.get(c, "#999999") for c in df.columns}
    plot_stacked_emissions(
        df,
        colors,
        axes[1, col],
        xlabel=xlabel,
        ylabel="Feed use [Gt DM]" if col == 0 else "",
        panel_label=panel_labels[panel_idx],
        x_ticks=x_ticks,
        x_ticklabels=x_ticklabels,
        min_height_for_label=0.3,
    )
    panel_idx += 1

# Column titles
axes[0, 0].set_title(
    "Health value sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)
axes[0, 1].set_title(
    "GHG price sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)
axes[0, 2].set_title(
    "Combined sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=10
)

# Share y-axis within rows
for row in range(2):
    y_min = min(ax.get_ylim()[0] for ax in axes[row, :])
    y_max = max(ax.get_ylim()[1] for ax in axes[row, :])
    for col in range(3):
        axes[row, col].set_ylim(y_min, y_max)

# Remove x-labels and ticks from top row
for col in range(3):
    axes[0, col].set_xlabel("")
    axes[0, col].set_xticklabels([])

# Dual x-axis labels on bottom-right
set_dual_xaxis_labels(
    axes[1, 2], GHG_YLL_XTICKS, GHG_YLL_GHG_VALUES, GHG_YLL_YLL_VALUES
)
set_dual_xlabel(axes[1, 2])

# Remove y-labels from cols 1 and 2
for row in range(2):
    for col in [1, 2]:
        axes[row, col].set_ylabel("")
        axes[row, col].set_yticklabels([])

fig.align_ylabels(axes[:, 0])
plt.tight_layout()
plt.subplots_adjust(bottom=0.12)

try:
    from IPython.display import display

    display(fig)
except Exception:
    plt.show()
plt.close(fig)

## Crop yield sensitivity figure


In [ ]:
# Crop yield panels by crop category (production-weighted mean yields)
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D


def normalize_to_initial_production(df):
    """Normalize production by each group's value in the initial scenario."""
    if df.empty:
        return df

    baseline = df.iloc[0].replace(0.0, np.nan)
    norm = df.divide(baseline, axis=1)

    # If initial production is zero, fall back to the first non-zero scenario.
    for group in norm.columns:
        if not np.isfinite(norm.at[norm.index[0], group]):
            nonzero = df[group][df[group] > 0]
            if nonzero.empty:
                norm[group] = 0.0
            else:
                norm[group] = df[group] / nonzero.iloc[0]

    return norm.replace([np.inf, -np.inf], np.nan).fillna(0.0)


def plot_crop_group_yield_panel(
    ax,
    yield_df,
    prod_norm_df,
    group_colors,
    xlabel,
    ylabel,
    panel_label,
    x_ticks,
    x_ticklabels,
):
    """Plot group yields with segment linewidth proportional to normalized production."""
    if yield_df.empty:
        return 0.0

    x_values = yield_df.index.values.astype(float)
    zero_pos = log_scale_zero_position(x_values)
    x_plot = np.where(x_values == 0, zero_pos, x_values)

    y_max = 0.0
    for group in yield_df.columns:
        y = yield_df[group].to_numpy(dtype=float)
        w = prod_norm_df[group].to_numpy(dtype=float)

        valid = np.isfinite(y) & np.isfinite(w)
        if valid.sum() < 2:
            continue

        xv = x_plot[valid]
        yv = y[valid]
        wv = np.clip(w[valid], 0.0, None)

        if np.nanmax(np.abs(yv)) < 1e-12 and np.nanmax(wv) < 1e-12:
            continue

        points = np.column_stack([xv, yv]).reshape(-1, 1, 2)
        segments = np.concatenate([points[:-1], points[1:]], axis=1)

        seg_w = 0.5 * (wv[:-1] + wv[1:])
        linewidths = 0.5 + 1.5 * np.clip(seg_w, 0, 3.0)

        collection = LineCollection(
            segments,
            colors=[group_colors.get(group, "#999999")],
            linewidths=linewidths,
            alpha=0.9,
            zorder=2,
        )
        ax.add_collection(collection)
        y_max = max(y_max, float(np.nanmax(yv)))

    x_max = max(float(np.max(x_plot)), max(x_ticks))
    x_min = min(zero_pos, min(x_ticks))

    ax.set_xscale("log")
    ax.set_xlim(x_min, x_max)
    ax.set_xticks(x_ticks)
    ax.set_xticklabels(x_ticklabels)
    ax.set_xlabel(xlabel, fontsize=FONTSIZE_AXIS_LABEL)
    ax.set_ylabel(ylabel, fontsize=FONTSIZE_AXIS_LABEL)
    ax.tick_params(axis="both", which="major", labelsize=FONTSIZE_TICK_LABEL)
    ax.tick_params(axis="both", which="minor", labelsize=FONTSIZE_TICK_LABEL)
    ax.grid(axis="y", alpha=0.25, linewidth=0.4)

    ax.text(
        0.02,
        0.98,
        panel_label,
        transform=ax.transAxes,
        fontsize=FONTSIZE_PANEL_LABEL,
        fontweight="bold",
        va="top",
    )

    return y_max


# Load crop-level yield and production data
yll_crop_yield, yll_crop_prod = load_crop_yield_and_production(
    yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "yll_value",
)
ghg_crop_yield, ghg_crop_prod = load_crop_yield_and_production(
    ghg_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "ghg_price",
)
ghg_yll_crop_yield, ghg_yll_crop_prod = load_crop_yield_and_production(
    ghg_yll_scenarios,
    PROJECT_ROOT,
    CONFIG_NAME,
    "ghg_price",
)

# Aggregate to crop groups using production-weighted mean yields
yll_group_yield, yll_group_prod = aggregate_crop_metrics_by_group(
    yll_crop_yield,
    yll_crop_prod,
)
ghg_group_yield, ghg_group_prod = aggregate_crop_metrics_by_group(
    ghg_crop_yield,
    ghg_crop_prod,
)
ghg_yll_group_yield, ghg_yll_group_prod = aggregate_crop_metrics_by_group(
    ghg_yll_crop_yield,
    ghg_yll_crop_prod,
)

# Consistent group order and colors from crop production map style
all_groups = sorted(
    set(yll_group_prod.columns)
    | set(ghg_group_prod.columns)
    | set(ghg_yll_group_prod.columns)
)
group_order = [g for g in CROP_GROUP_COLORS if g in all_groups] + [
    g for g in all_groups if g not in CROP_GROUP_COLORS
]
group_colors = {g: CROP_GROUP_COLORS.get(g, "#999999") for g in group_order}

# Align columns
yll_group_yield = yll_group_yield.reindex(columns=group_order, fill_value=0.0)
ghg_group_yield = ghg_group_yield.reindex(columns=group_order, fill_value=0.0)
ghg_yll_group_yield = ghg_yll_group_yield.reindex(columns=group_order, fill_value=0.0)
yll_group_prod = yll_group_prod.reindex(columns=group_order, fill_value=0.0)
ghg_group_prod = ghg_group_prod.reindex(columns=group_order, fill_value=0.0)
ghg_yll_group_prod = ghg_yll_group_prod.reindex(columns=group_order, fill_value=0.0)

# Normalize line width data to initial scenario for each sensitivity set
yll_group_prod_norm = normalize_to_initial_production(yll_group_prod)
ghg_group_prod_norm = normalize_to_initial_production(ghg_group_prod)
ghg_yll_group_prod_norm = normalize_to_initial_production(ghg_yll_group_prod)

# Build figure: 1 row x 3 columns
fig, axes = plt.subplots(1, 3, figsize=(8, 3.2), sharey=True)

ymax_values = []
ymax_values.append(
    plot_crop_group_yield_panel(
        axes[0],
        yll_group_yield,
        yll_group_prod_norm,
        group_colors,
        xlabel=YLL_XLABEL,
        ylabel="Average yield [Mt/Mha]",
        panel_label="a",
        x_ticks=YLL_XTICKS,
        x_ticklabels=YLL_XTICKLABELS,
    )
)
ymax_values.append(
    plot_crop_group_yield_panel(
        axes[1],
        ghg_group_yield,
        ghg_group_prod_norm,
        group_colors,
        xlabel=GHG_XLABEL,
        ylabel="",
        panel_label="b",
        x_ticks=GHG_XTICKS,
        x_ticklabels=GHG_XTICKLABELS,
    )
)
ymax_values.append(
    plot_crop_group_yield_panel(
        axes[2],
        ghg_yll_group_yield,
        ghg_yll_group_prod_norm,
        group_colors,
        xlabel="",
        ylabel="",
        panel_label="c",
        x_ticks=GHG_YLL_XTICKS,
        x_ticklabels=[""] * len(GHG_YLL_XTICKS),
    )
)

y_max = max(ymax_values) if ymax_values else 0.0
if y_max > 0:
    for ax in axes:
        ax.set_ylim(0, y_max * 1.1)

axes[0].set_title(
    "Health value sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=8
)
axes[1].set_title(
    "GHG price sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=8
)
axes[2].set_title(
    "Combined sensitivity", fontsize=FONTSIZE_TITLE + 2, fontweight="bold", pad=8
)

set_dual_xaxis_labels(axes[2], GHG_YLL_XTICKS, GHG_YLL_GHG_VALUES, GHG_YLL_YLL_VALUES)
set_dual_xlabel(axes[2])

# Legend order: by combined-panel yield at the left-most scenario
if not ghg_yll_group_yield.empty:
    legend_order = (
        ghg_yll_group_yield.iloc[0].sort_values(ascending=False).index.tolist()
    )
else:
    legend_order = group_order

legend_handles = [
    Line2D([0], [0], color=group_colors[group], lw=2, label=group)
    for group in legend_order
    if group in group_colors
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    ncol=4,
    frameon=False,
    fontsize=FONTSIZE_TICK_LABEL,
    bbox_to_anchor=(0.5, -0.10),
)

plt.tight_layout()
plt.subplots_adjust(bottom=0.28)
try:
    from IPython.display import display

    display(fig)
except Exception:
    plt.show()
plt.close(fig)